In [0]:
# ============================================================
# Maven Market Lakehouse — CI/CD Test Suite
# 97 Tests: Bronze (28) + Silver (31) + Gold (38)
# ============================================================

CATALOG = "maven_catalog"
BRONZE  = "bronze_schema"
SILVER  = "silver_schema"
GOLD    = "gold_schema"

def t(schema, name):
    return spark.read.table(f"{CATALOG}.{schema}.{name}")

results = {"passed": 0, "failed": 0, "errors": []}

def check(test_name, condition, msg=""):
    if condition:
        results["passed"] += 1
        print(f"  ✅ {test_name}")
    else:
        results["failed"] += 1
        results["errors"].append(f"{test_name}: {msg}")
        print(f"  ❌ {test_name} — {msg}")

def safe_check(test_name, fn):
    try:
        ok, msg = fn()
        check(test_name, ok, msg)
    except Exception as e:
        results["failed"] += 1
        results["errors"].append(f"{test_name}: {e}")
        print(f"  ❌ {test_name} — ERROR: {e}")

# ============================================================
# BRONZE TESTS
# ============================================================
print("=" * 60)
print("BRONZE LAYER TESTS")
print("=" * 60)

bronze_tables = {
    "brz_transactions": 200000,
    "brz_returns": 7000,
    "brz_stores": 20,
    "brz_regions": 100,
    "brz_calendar": 300,
    "brz_products_mongo_dlt": 1500,
    "brz_customers_mongo_dlt": 10000,
    "brz_kafka_orders_raw": 1,
    "brz_kafka_orders": 1,
    "brz_kafka_inventory_raw": 1,
    "brz_kafka_inventory": 1,
}

# B1. Existence & Row Counts
for tbl, min_rows in bronze_tables.items():
    safe_check(f"BRZ row count: {tbl} >= {min_rows}",
        lambda tbl=tbl, m=min_rows: (
            (lambda c: (c >= m, f"got {c}"))(t(BRONZE, tbl).count())
        ))
# B2. Schema — critical columns
brz_schema_checks = {
    "brz_transactions": ["transaction_date", "product_id", "customer_id", "store_id", "quantity"],
    "brz_returns": ["return_date", "product_id", "store_id", "quantity"],
    "brz_stores": ["store_id", "region_id", "store_name"],
    "brz_products_mongo_dlt": ["product_id", "product_name", "product_retail_price", "product_cost"],
    "brz_customers_mongo_dlt": ["customer_id", "first_name", "last_name", "customer_country", "gender"],
    "brz_kafka_orders": ["order_id", "product_id", "customer_id", "store_id", "quantity", "total_amount", "event_time"],
    "brz_kafka_inventory": ["current_stock", "quantity_change", "change_type", "product_id", "store_id", "event_time"],
}
for tbl, expected_cols in brz_schema_checks.items():
    safe_check(f"BRZ schema: {tbl}",
        lambda tbl=tbl, ec=expected_cols: (
            (lambda cols: (all(c in cols for c in ec), f"missing: {[c for c in ec if c not in cols]}"))(t(BRONZE, tbl).columns)
        ))

# B3. Metadata columns
for tbl in ["brz_transactions", "brz_returns", "brz_stores", "brz_regions", "brz_calendar",
            "brz_products_mongo_dlt", "brz_customers_mongo_dlt", "brz_kafka_orders", "brz_kafka_inventory"]:
    safe_check(f"BRZ metadata: {tbl}",
        lambda tbl=tbl: (
            (lambda cols: ("_ingestion_timestamp" in cols and "_source_system" in cols,
                           f"missing metadata cols"))(t(BRONZE, tbl).columns)
        ))

# B4. Data quality
safe_check("BRZ quality: products price > 0",
    lambda: (t(BRONZE, "brz_products_mongo_dlt").filter("product_retail_price <= 0").count() == 0, "bad prices found"))
safe_check("BRZ quality: customers valid gender",
    lambda: (t(BRONZE, "brz_customers_mongo_dlt").filter("gender NOT IN ('M','F') AND gender IS NOT NULL").count() == 0, "invalid genders"))
safe_check("BRZ quality: kafka inventory stock >= 0",
    lambda: (t(BRONZE, "brz_kafka_inventory").filter("current_stock < 0").count() == 0, "negative stock"))

# B5. No duplicates in dimensions
for tbl, key in [("brz_stores", "store_id"), ("brz_regions", "region_id"), ("brz_products_mongo_dlt", "product_id")]:
    safe_check(f"BRZ no dupes: {tbl}.{key}",
        lambda tbl=tbl, k=key: (
            (lambda df: (df.count() == df.dropDuplicates([k]).count(), f"duplicates found"))(t(BRONZE, tbl))
        ))

# ============================================================
# SILVER TESTS
# ============================================================
print("\n" + "=" * 60)
print("SILVER LAYER TESTS")
print("=" * 60)

# S1. Table existence
silver_tables = ["slv_transactions", "slv_returns", "slv_stores", "slv_regions",
                 "slv_calendar", "slv_customers", "slv_kafka_orders", "slv_kafka_inventory"]
for tbl in silver_tables:
    safe_check(f"SLV exists: {tbl}",
        lambda tbl=tbl: (t(SILVER, tbl).count() > 0, "empty table"))

# S2. Quarantine tables exist
quarantine_tables = ["quarantine_transactions", "quarantine_returns",
                     "quarantine_kafka_orders", "quarantine_kafka_inventory"]
for tbl in quarantine_tables:
    safe_check(f"SLV quarantine exists: {tbl}",
        lambda tbl=tbl: (t(SILVER, tbl) is not None, "table not found"))

# S3. Schema checks
slv_schema_checks = {
    "slv_transactions": ["transaction_date", "product_id", "customer_id", "store_id", "quantity"],
    "slv_customers": ["customer_id", "__START_AT", "__END_AT"],
    "slv_stores": ["store_id", "store_name", "sales_district", "sales_region"],
    "slv_calendar": ["date", "year", "month", "quarter", "day_of_week", "is_weekend"],
    "slv_kafka_orders": ["order_id", "total_amount", "product_id", "customer_id", "store_id", "quantity", "event_time"],
    "slv_kafka_inventory": ["current_stock", "change_type", "product_id", "store_id", "stock_status", "is_low_stock"],
}
for tbl, expected_cols in slv_schema_checks.items():
    safe_check(f"SLV schema: {tbl}",
        lambda tbl=tbl, ec=expected_cols: (
            (lambda cols: (all(c in cols for c in ec), f"missing: {[c for c in ec if c not in cols]}"))(t(SILVER, tbl).columns)
        ))

# S4. Quarantine metadata
for tbl in quarantine_tables:
    safe_check(f"SLV quarantine metadata: {tbl}",
        lambda tbl=tbl: (
            (lambda cols: ("_quarantine_reasons" in cols and "_quarantine_timestamp" in cols,
                           f"missing quarantine cols"))(t(SILVER, tbl).columns)
        ))

# S5. Null checks on clean tables
for col_name in ["product_id", "customer_id", "store_id"]:
    safe_check(f"SLV no nulls: slv_transactions.{col_name}",
        lambda c=col_name: (t(SILVER, "slv_transactions").filter(f"{c} IS NULL").count() == 0, "nulls found"))

safe_check("SLV quality: slv_transactions quantity > 0",
    lambda: (t(SILVER, "slv_transactions").filter("quantity <= 0").count() == 0, "bad quantities"))
safe_check("SLV quality: slv_kafka_inventory stock >= 0",
    lambda: (t(SILVER, "slv_kafka_inventory").filter("current_stock < 0").count() == 0, "negative stock"))

# S6. SCD Type 2

# S6. SCD Type 2 — using __END_AT IS NULL for current records (no __IS_CURRENT in your table)
safe_check("SLV SCD2: has current records (__END_AT IS NULL)",
    lambda: (t(SILVER, "slv_customers").filter("__END_AT IS NULL").count() > 0, "no current records"))
safe_check("SLV SCD2: current rows unique per customer_id",
    lambda: (
        (lambda df: (df.count() == df.select("customer_id").distinct().count(), "duplicate current rows"))(
            t(SILVER, "slv_customers").filter("__END_AT IS NULL")
        )))
safe_check("SLV SCD2: __START_AT not null for current rows",
    lambda: (t(SILVER, "slv_customers").filter("__END_AT IS NULL AND __START_AT IS NULL").count() == 0, "null START_AT"))

# S7. No duplicates
for tbl, key in [("slv_stores", "store_id"), ("slv_regions", "region_id")]:
    safe_check(f"SLV no dupes: {tbl}.{key}",
        lambda tbl=tbl, k=key: (
            (lambda df: (df.count() == df.dropDuplicates([k]).count(), "duplicates"))(t(SILVER, tbl))
        ))

# S8. Bronze-Silver-Quarantine balance
safe_check("SLV balance: brz_txn = slv_txn + quarantine_txn (±5%)",
    lambda: (
        (lambda b, s, q: (abs(b - (s + q)) / b * 100 < 5 if b > 0 else True,
                          f"brz={b}, slv={s}, q={q}"))(
            t(BRONZE, "brz_transactions").count(),
            t(SILVER, "slv_transactions").count(),
            t(SILVER, "quarantine_transactions").count()
        )))

# S9. Enrichment
safe_check("SLV enrichment: slv_stores.sales_region not null",
    lambda: (t(SILVER, "slv_stores").filter("sales_region IS NULL").count() == 0, "null sales_region"))
safe_check("SLV enrichment: stock_status valid values",
    lambda: (
        t(SILVER, "slv_kafka_inventory")
        .filter("stock_status NOT IN ('Out of Stock','Low Stock','Normal','Well Stocked')")
        .count() == 0, "invalid stock_status"))

# ============================================================
# GOLD TESTS
# ============================================================
print("\n" + "=" * 60)
print("GOLD LAYER TESTS")
print("=" * 60)

# G1. Table existence — test each individually so one failure doesn't block all
gold_tables = [
    "dim_customers", "dim_products", "dim_stores", "dim_calendar",
    "fact_sales", "fact_returns", "fact_kafka_orders", "fact_inventory_movements",
    "agg_daily_sales", "agg_monthly_sales", "agg_product_performance",
    "agg_store_performance", "agg_kafka_throughput_min",
    "kpi_executive_summary", "gold_pipeline_health_audit",
]
for tbl in gold_tables:
    safe_check(f"GLD exists: {tbl}",
        lambda tbl=tbl: (t(GOLD, tbl).count() > 0, "empty"))
# After the G1 loop, add:
safe_check("GLD exists: agg_inventory_alerts (can be empty)",
    lambda: (t(GOLD, "agg_inventory_alerts") is not None, "table not found"))

# G2. Dimension schemas
gld_schema_checks = {
    "dim_customers": ["customer_id", "full_name", "customer_country", "member_card", "gender"],
    "dim_products": ["product_id", "product_brand", "product_name", "product_retail_price", "profit_margin", "price_tier"],
    "dim_stores": ["store_id", "store_name", "store_country", "sales_region", "total_sqft"],
    "dim_calendar": ["date", "year", "month", "quarter", "is_weekend"],
    "fact_sales": ["transaction_date", "product_id", "customer_id", "store_id", "quantity", "total_revenue", "total_cost", "total_profit"],
    "fact_returns": ["return_date", "product_id", "store_id", "quantity"],
    "fact_kafka_orders": ["order_id", "quantity", "event_time"],
}
for tbl, expected_cols in gld_schema_checks.items():
    safe_check(f"GLD schema: {tbl}",
        lambda tbl=tbl, ec=expected_cols: (
            (lambda cols: (all(c in cols for c in ec), f"missing: {[c for c in ec if c not in cols]}"))(t(GOLD, tbl).columns)
        ))

# G3. Dimension uniqueness
for tbl, key in [("dim_customers", "customer_id"), ("dim_products", "product_id"),
                  ("dim_stores", "store_id"), ("dim_calendar", "date")]:
    safe_check(f"GLD unique: {tbl}.{key}",
        lambda tbl=tbl, k=key: (
            (lambda df: (df.count() == df.select(k).distinct().count(), "duplicates"))(t(GOLD, tbl))
        ))

# G4. Fact data quality
safe_check("GLD quality: fact_sales revenue > 0",
    lambda: (
        (lambda bad, total: (bad / total * 100 < 1 if total > 0 else True, f"{bad}/{total} bad rows"))(
            t(GOLD, "fact_sales").filter("total_revenue <= 0").count(),
            t(GOLD, "fact_sales").count()
        )))
safe_check("GLD quality: fact_sales no null keys",
    lambda: (
        t(GOLD, "fact_sales").filter("product_id IS NULL OR customer_id IS NULL OR store_id IS NULL").count() == 0,
        "null FKs found"))

# G5. Referential integrity (FK → Dimension)
fk_checks = [
    ("fact_sales", "product_id", "dim_products", "product_id"),
    ("fact_sales", "customer_id", "dim_customers", "customer_id"),
    ("fact_sales", "store_id", "dim_stores", "store_id"),
    ("fact_returns", "product_id", "dim_products", "product_id"),
    ("fact_returns", "store_id", "dim_stores", "store_id"),
]
for fact_tbl, fact_col, dim_tbl, dim_col in fk_checks:
    safe_check(f"GLD FK: {fact_tbl}.{fact_col} → {dim_tbl}",
        lambda ft=fact_tbl, fc=fact_col, dt=dim_tbl, dc=dim_col: (
            (lambda orphans: (orphans == 0, f"{orphans} orphan keys"))(
                t(GOLD, ft).select(fc).distinct().subtract(
                    t(GOLD, dt).select(dc).distinct()
                ).count()
            )))

# G6. Aggregate accuracy
safe_check("GLD agg: daily_sales revenue ≈ fact_sales revenue (±1%)",
    lambda: (
        (lambda a, f: (abs(a - f) / f * 100 < 1 if f > 0 else True,
                       f"agg={a:.0f}, fact={f:.0f}"))(
            float(t(GOLD, "agg_daily_sales").agg({"total_revenue": "sum"}).collect()[0][0] or 0),
            float(t(GOLD, "fact_sales").agg({"total_revenue": "sum"}).collect()[0][0] or 0)
        )))

safe_check("GLD agg: agg_product_performance has sales_region",
    lambda: ("sales_region" in t(GOLD, "agg_product_performance").columns, "missing sales_region"))

# G7. KPI validation
# G7. KPI validation — only test columns that exist
safe_check("GLD KPI: profit_margin 0-100%",
    lambda: (t(GOLD, "kpi_executive_summary").filter("profit_margin_pct < 0 OR profit_margin_pct > 100").count() == 0, "invalid margin"))
safe_check("GLD KPI: total_revenue > 0",
    lambda: (t(GOLD, "kpi_executive_summary").filter("total_revenue <= 0").count() == 0, "invalid revenue"))

# G8. Cross-layer row flow
safe_check("GLD cross-layer: fact_sales ≈ slv_transactions (±5%)",
    lambda: (
        (lambda g, s: (abs(g - s) / s * 100 < 5 if s > 0 else True,
                       f"gold={g}, silver={s}"))(
            t(GOLD, "fact_sales").count(),
            t(SILVER, "slv_transactions").count()
        )))

# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "=" * 60)
print(f"RESULTS: {results['passed']} PASSED | {results['failed']} FAILED")
print("=" * 60)

if results["failed"] > 0:
    print("\n⚠️  FAILURES:")
    for err in results["errors"]:
        print(f"   • {err}")
    print()
else:
    print("\n🎉 ALL TESTS PASSED — Pipeline is healthy!")
